# Tempo Run 2025 — Download data từ HF + Build SFT JSONL

Mục đích: tải data từ Hugging Face private dataset, build train/eval JSONL.
Phụ thuộc: notebook `00_setup_colab.ipynb` đã chạy và `.env` đã có `HF_TOKEN` + `HF_DATASET_REPO`.

In [ ]:
%cd /content/finetune_1B_MCQ_VN
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), 'src'))
from temprun.utils import load_env, repo_root
load_env(repo_root() / ".env")

In [ ]:
# 1. Tải data từ HF private dataset (~5-10s cho 6MB)
!python scripts/download_data.py

In [ ]:
# 2. Verify
!ls data/raw/
!echo '---'
!echo "train files: $(ls data/raw/train | wc -l)"
!echo "test  files: $(ls data/raw/test  | wc -l)"
!ls data/raw/sample_submission.csv 2>/dev/null && echo "sample_submission.csv: OK"

In [ ]:
# 3. Build SFT jsonl (90/10 stratified split)
!python scripts/make_sft_jsonl.py --in data/raw/train --out data/processed

In [ ]:
# 4. Inspect 1 row để chắc chắn schema đúng
import json
with open('data/processed/train.jsonl') as f:
    r = json.loads(f.readline())
print('keys:', list(r.keys()))
print('label:', r['label'])
print('row_id:', r.get('row_id'))
for m in r['messages']:
    print(f"--- {m['role']} ---")
    print(m['content'][:400] + ('...' if len(m['content']) > 400 else ''))